# FraudeCero · Grupo 6 · 05 · Monitoreo y optimización (A3.4)
**Proyecto Integrador Big Data – Uniciencia 2026-II** · Integrantes: Juan David Montoya · William Conde Villamizar


**Objetivo:** tablero operativo con **4 métricas** a partir del log que escribe cada notebook (`99_evidencias/monitoreo/log_ejecuciones.csv`):

| Métrica | Columna del log | Para qué sirve |
|---|---|---|
| M1 · Tiempo de ejecución | `duracion_s` | Detectar etapas que se vuelven lentas |
| M2 · Filas procesadas | `filas` | Detectar cargas incompletas (como el CSV cortado de la primera prueba) |
| M3 · % de nulos | `pct_nulos` | Detectar degradación de la calidad de la fuente |
| M4 · Consumo de recursos | `memoria_mb`, `cpu_pct` | Dimensionar el cómputo |

**Job programado:** `.github/workflows/pipeline_fraudecero.yml` (GitHub Actions) ejecuta el pipeline todos los días a las 06:17 (hora Colombia) y manualmente; cada ejecución agrega filas a este log con su `run_id`.

**Regla de alerta (escrita con su umbral):**
> **ALERTA** si en una etapa (a) `duracion_s` > **2 ×** la mediana histórica de esa etapa (con ≥ 2 ejecuciones previas), o (b) `pct_nulos` > **1 %**, o (c) `filas` < **99 %** de la mediana histórica de esa etapa.

## Configuración común
Esta celda es igual en todos los notebooks. Detecta si se ejecuta en **Google Colab** o en **Jupyter / GitHub Actions**, instala PySpark 3.5 + Delta Lake (OSS) y define:
- `LAKE`: carpeta del Lakehouse (capas `bronce`, `plata`, `oro`).
- `EVID`: carpeta `99_evidencias` (tablas de evidencia, log de monitoreo, tableros HTML).
- `guardar()` / `leer()`: escriben y leen tablas **Delta** (si Delta no está disponible usan Parquet y lo avisan).
- `registrar()`: escribe en el log de monitoreo el tiempo, las filas, el % de nulos y la memoria de cada etapa.

> En Colab cada notebook corre en una máquina distinta, por eso el Lakehouse se respalda en Google Drive al final de cada notebook y se restaura al inicio del siguiente.

In [ ]:
# ===================== CONFIGURACIÓN COMÚN FRAUDECERO =====================
NOTEBOOK = "05_monitoreo"
import os, sys, time, json, shutil, datetime, subprocess, hashlib
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub", "psutil", "plotly"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DRIVE = "/content/drive/MyDrive/FraudeCero"
    LAKE = "/content/lakehouse"                      # disco local (rápido)
    LAKE_RESPALDO = f"{BASE_DRIVE}/lakehouse"        # respaldo persistente en Drive
    EVID = f"{BASE_DRIVE}/99_evidencias"
else:
    RAIZ = os.environ.get("FRAUDECERO_RAIZ", os.path.abspath(".."))
    LAKE = os.environ.get("FRAUDECERO_LAKE", os.path.join(RAIZ, "lakehouse"))
    LAKE_RESPALDO = None
    EVID = os.environ.get("FRAUDECERO_EVID", os.path.join(RAIZ, "99_evidencias"))

for carpeta in [LAKE, EVID, f"{EVID}/monitoreo", f"{EVID}/tablas"]:
    os.makedirs(carpeta, exist_ok=True)

def restaurar_lake():
    """En Colab: trae el Lakehouse desde Drive si el disco local está vacío."""
    if LAKE_RESPALDO and os.path.exists(LAKE_RESPALDO) and not os.listdir(LAKE):
        shutil.copytree(LAKE_RESPALDO, LAKE, dirs_exist_ok=True)
        print("Lakehouse restaurado desde Drive")

def respaldar_lake():
    """En Colab: copia el Lakehouse a Drive (sin la zona landing de archivos crudos)."""
    if LAKE_RESPALDO:
        shutil.copytree(LAKE, LAKE_RESPALDO, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("landing"))
        print("Lakehouse respaldado en Drive:", LAKE_RESPALDO)

restaurar_lake()

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

FORMATO = "delta"
try:
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
               .config("spark.driver.memory", "6g")
               .config("spark.sql.shuffle.partitions", "8")
               .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
               .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.range(1).write.format("delta").mode("overwrite").save(f"{LAKE}/_prueba_delta")
except Exception as e:
    print("⚠️ Delta Lake no disponible, se usará Parquet. Detalle:", str(e)[:300])
    FORMATO = "parquet"
    spark = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
             .config("spark.driver.memory", "6g")
             .config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def ruta(capa, tabla):
    return f"{LAKE}/{capa}/{tabla}"

def leer(capa, tabla):
    df = spark.read.format(FORMATO).load(ruta(capa, tabla))
    df.createOrReplaceTempView(f"{capa}_{tabla}")
    return df

def guardar(df, capa, tabla, particion=None):
    w = df.write.format(FORMATO).mode("overwrite").option("overwriteSchema", "true")
    if particion:
        w = w.partitionBy(particion)
    w.save(ruta(capa, tabla))
    return leer(capa, tabla)

# ---------- Monitoreo: tiempo, filas, % nulos y memoria por etapa ----------
import psutil, pandas as pd
LOG_MON = f"{EVID}/monitoreo/log_ejecuciones.csv"
ARCHIVO_RUN = f"{EVID}/monitoreo/run_actual.txt"
RUN_ID = os.environ.get("FRAUDECERO_RUN_ID")
if not RUN_ID:
    if NOTEBOOK == "00_ingesta" or not os.path.exists(ARCHIVO_RUN):
        RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")   # cada corrida del pipeline empieza en 00
        with open(ARCHIVO_RUN, "w") as f:
            f.write(RUN_ID)
    else:
        RUN_ID = open(ARCHIVO_RUN).read().strip()
ORIGEN = "github_actions" if os.environ.get("GITHUB_ACTIONS") else ("colab" if EN_COLAB else "jupyter")

def memoria_mb():
    py = psutil.Process(os.getpid()).memory_info().rss / 1024**2
    try:
        rt = spark.sparkContext._jvm.java.lang.Runtime.getRuntime()
        jvm = (rt.totalMemory() - rt.freeMemory()) / 1024**2
    except Exception:
        jvm = 0.0
    return round(py + jvm, 1)

def registrar(etapa, t0, df=None, filas=None):
    """Registra una etapa en el log de monitoreo. t0 = time.time() al inicio de la etapa."""
    duracion = round(time.time() - t0, 2)
    pct_nulos = None
    if df is not None:
        filas = df.count()
        cols = df.columns
        nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).collect()[0]
        pct_nulos = round(sum(v or 0 for v in nulos) / (filas * len(cols)) * 100, 4) if filas else 0.0
    fila = {"run_id": RUN_ID, "origen": ORIGEN, "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            "notebook": NOTEBOOK, "etapa": etapa, "duracion_s": duracion, "filas": filas,
            "pct_nulos": pct_nulos, "memoria_mb": memoria_mb(), "cpu_pct": psutil.cpu_percent(interval=0.2),
            "formato": FORMATO}
    pd.DataFrame([fila]).to_csv(LOG_MON, mode="a", index=False, header=not os.path.exists(LOG_MON))
    print(f"⏱️ {etapa}: {duracion} s · filas={filas} · nulos={pct_nulos}% · mem={fila['memoria_mb']} MB")

def mostrar(df, n=20):
    """display() de Databricks no existe en Jupyter/Colab: se muestra como tabla pandas."""
    from IPython.display import display
    display(df.limit(n).toPandas() if hasattr(df, "toPandas") else df)

print(f"Notebook: {NOTEBOOK} · Entorno: {ORIGEN} · Formato de tablas: {FORMATO.upper()} · Spark {spark.version}")
print("LAKE =", LAKE, "| EVID =", EVID)

In [ ]:
import plotly.express as px
log = pd.read_csv(LOG_MON)
log["timestamp"] = pd.to_datetime(log["timestamp"])
print(f"Ejecuciones registradas: {log.run_id.nunique()} · etapas registradas: {len(log)}")
resumen = (log.groupby(["run_id", "origen"]).agg(inicio=("timestamp", "min"), fin=("timestamp", "max"),
           etapas=("etapa", "count"), duracion_total_s=("duracion_s", "sum"), memoria_max_mb=("memoria_mb", "max"))
           .reset_index().sort_values("inicio"))
mostrar(resumen)

## Regla de alerta

In [ ]:
UMBRAL_TIEMPO, UMBRAL_NULOS, UMBRAL_FILAS = 2.0, 1.0, 0.99
alertas = []
ultimo = resumen.run_id.iloc[-1]
for _, fila in log[log.run_id == ultimo].iterrows():
    hist = log[(log.etapa == fila.etapa) & (log.run_id != ultimo)]
    if len(hist) >= 2 and fila.duracion_s > UMBRAL_TIEMPO * hist.duracion_s.median():
        alertas.append((fila.etapa, "tiempo", f"{fila.duracion_s:.1f} s > {UMBRAL_TIEMPO}× mediana ({hist.duracion_s.median():.1f} s)"))
    if pd.notna(fila.pct_nulos) and fila.pct_nulos > UMBRAL_NULOS:
        alertas.append((fila.etapa, "nulos", f"{fila.pct_nulos:.2f} % > {UMBRAL_NULOS} %"))
    hist_f = hist.filas.dropna()
    if pd.notna(fila.filas) and len(hist_f) >= 2 and fila.filas < UMBRAL_FILAS * hist_f.median():
        alertas.append((fila.etapa, "filas", f"{fila.filas:,.0f} < {UMBRAL_FILAS:.0%} de la mediana ({hist_f.median():,.0f})"))

tabla_alertas = pd.DataFrame(alertas, columns=["etapa", "tipo", "detalle"])
tabla_alertas.insert(0, "run_id", ultimo)
tabla_alertas.to_csv(f"{EVID}/monitoreo/alertas_{ultimo}.csv", index=False)
print("🚨 ALERTAS:" if alertas else "✅ Sin alertas en la última ejecución", ultimo)
mostrar(tabla_alertas)

## Tablero operativo (4 métricas)

In [ ]:
etapas_clave = ["bronce_transacciones", "plata_transacciones", "T6_union_fuentes", "modelo_entrenamiento", "tablero_html"]
fig_m1 = px.bar(log.groupby(["run_id", "notebook"]).duracion_s.sum().reset_index(), x="run_id", y="duracion_s", color="notebook",
                title="M1 · Tiempo de ejecución por notebook (s)", labels={"duracion_s": "segundos", "run_id": "ejecución"})
log_filas = log.dropna(subset=["filas"])
log_filas = log_filas[log_filas.etapa.isin(etapas_clave) | log_filas.etapa.str.startswith("bronce")]
fig_m2 = px.line(log_filas,
                 x="run_id", y="filas", color="etapa", markers=True,
                 title="M2 · Filas procesadas por etapa", labels={"filas": "filas", "run_id": "ejecución"})
fig_m3 = px.line(log.dropna(subset=["pct_nulos"]), x="run_id", y="pct_nulos", color="etapa", markers=True,
                 title="M3 · % de nulos por tabla (umbral de alerta = 1 %)", labels={"pct_nulos": "% nulos", "run_id": "ejecución"})
fig_m3.add_hline(y=UMBRAL_NULOS, line_dash="dash", line_color="red")
fig_m4 = px.line(log.groupby("run_id").agg(memoria_mb=("memoria_mb", "max"), cpu_pct=("cpu_pct", "mean")).reset_index()
                 .melt(id_vars="run_id"), x="run_id", y="value", color="variable", markers=True,
                 title="M4 · Consumo de recursos (memoria máx. MB · CPU media %)", labels={"value": "valor", "run_id": "ejecución"})
html = ["<html><head><meta charset='utf-8'><title>FraudeCero · Monitoreo</title></head><body style='font-family:Arial;max-width:1100px;margin:auto'>",
        f"<h1>FraudeCero · Tablero operativo</h1><p>Ejecuciones: {log.run_id.nunique()} · Última: {ultimo} · "
        f"Estado: {'🚨 ' + str(len(alertas)) + ' alerta(s)' if alertas else '✅ sin alertas'}</p>",
        "<p><b>Regla:</b> alerta si tiempo &gt; 2× mediana histórica, nulos &gt; 1 % o filas &lt; 99 % de la mediana histórica.</p>"]
for i, f in enumerate([fig_m1, fig_m2, fig_m3, fig_m4]):
    f.update_layout(height=380)
    f.show()
    html.append(f.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False))
html.append(tabla_alertas.to_html(index=False) if alertas else "")
html.append("</body></html>")
with open(f"{EVID}/monitoreo/tablero_operativo.html", "w", encoding="utf-8") as f:
    f.write("\n".join(html))
print("Tablero operativo:", f"{EVID}/monitoreo/tablero_operativo.html")

## Cierre
Respalda el Lakehouse (en Colab) y confirma la ejecución completa.

In [ ]:
respaldar_lake()
print("✅ Notebook", NOTEBOOK, "terminado · run_id:", RUN_ID)